# 03. PROJEKT — graf miast, próg sklejenia i screening połączeń lokalnych

**Czas na rozpoczęcie podczas zajęć:** około 3 godzin  
**Dane:** realne współrzędne miejscowości z przetworzonego zbioru PRNG.

## Główna historia projektu

Wykorzystujemy jeden logiczny ciąg:

$$
X
\longrightarrow
D
\longrightarrow
A_\varepsilon\ \text{lub}\ A_{kNN}
\longrightarrow
W
\longrightarrow
G
\longrightarrow
\text{DBSCAN / huby / dostępność / modele drzewiaste}.
$$

Projekt odpowiada na cztery praktyczne pytania:

1. **Skala sieci:** przy jakim promieniu lokalne skupiska miast zaczynają tworzyć jedną dużą sieć?
2. **Definicja sąsiedztwa:** co zmienia wybór grafu $\varepsilon$ w porównaniu z grafem $kNN$?
3. **Huby i peryferia:** które miasta oraz gminy są geometrycznie centralne, a które odległe od centrów?
4. **Screening transportowy:** które relacje gmina $\rightarrow$ miasto warto zweryfikować dodatkowymi danymi?

## Hipotezy, które będziemy sprawdzać

Na obecnym zbiorze referencyjnym powinny pojawić się czytelne efekty:

- największa składowa grafu miast rośnie bardzo szybko w okolicy kilkunastu–dwudziestu kilku kilometrów,
- pełna spójność wymaga znacznie większego progu niż pojawienie się „gigantycznej” składowej,
- ranking stopni powinien wskazać zwartą konurbację górnośląską,
- graf $kNN$ zapewnia sąsiadów także miastom peryferyjnym, ale może tworzyć bardzo długie krawędzie,
- DBSCAN daje wiele lokalnych skupisk dla małego `eps`, a następnie skleja je wraz ze wzrostem skali.

## Minimalny rezultat studenta

- wykonane wszystkie obowiązkowe `TODO`,
- wybrane jedno województwo,
- minimum 5–10 konkretnych obserwacji,
- rozróżnienie wyniku geometrycznego od realnego popytu,
- wskazanie danych potrzebnych do decyzji planistycznej.

> `need_score` nie jest prognozą liczby pasażerów. Jest wskaźnikiem screeningowym opartym na liczbie punktów osadniczych i odległościach w linii prostej.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, minimum_spanning_tree
from scipy.stats import spearmanr
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import haversine_distances, pairwise_distances
from sklearn.neighbors import BallTree
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, Isomap, trustworthiness
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, accuracy_score

R_EARTH_KM = 6371.0088
RANDOM_STATE = 42
EPS_NUMERIC = 1e-12
pd.set_option("display.max_columns", 50)

def locate_data(filename):
    candidates = [
        Path("data") / filename,
        Path(filename),
        Path.cwd() / "data" / filename,
        Path.cwd().parent / "data" / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Nie znaleziono {filename}. Sprawdzone ścieżki: {candidates}"
    )

def gmina_key(df):
    return (
        df["wojewodztwo"].astype(str) + "|"
        + df["powiat"].astype(str) + "|"
        + df["gmina"].astype(str)
    )

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False


## 1. Wczytanie i audyt danych

Jeden wiersz oznacza nazwaną miejscowość lub miasto, a nie osobę ani jednostkę popytu.

Najważniejsze kolumny:

- `name`,
- `place_type`,
- `gmina`, `powiat`, `wojewodztwo`,
- `lat`, `lon`.

Pierwsze pytania:

1. Ile jest wszystkich punktów?
2. Ile jest miast?
3. Czy brakuje współrzędnych?
4. Czy punktów PRNG można używać jako zamiennika populacji? — **nie**.

In [ ]:
DATA_PATH = locate_data("miejscowosci_polska_PRNG_2025_44631.csv")
places = pd.read_csv(DATA_PATH)
places["gmina_key"] = gmina_key(places)

print("Plik:", DATA_PATH)
print("Wymiary:", places.shape)
print("Braki lat/lon:", places[["lat", "lon"]].isna().sum().to_dict())
display(places.head())
display(places["place_type"].value_counts())

cities = places[places["place_type"].eq("city")].copy().reset_index(drop=True)
noncities = places[~places["place_type"].eq("city")].copy().reset_index(drop=True)

print("Miasta:", len(cities))
print("Pozostałe miejscowości:", len(noncities))
print("Opis źródła z pliku:")
print(places["source"].iloc[0])

# Część A — graf miast Polski

## 2. `TODO 1` — macierz odległości geograficznych $D_{\text{city}}$

Dla współrzędnych geograficznych używamy odległości haversine, a nie zwykłej odległości euklidesowej na stopniach.

$$
d =
2R\arcsin
\sqrt{
\sin^2\left(\frac{\Delta\varphi}{2}\right)
+
\cos\varphi_1\cos\varphi_2
\sin^2\left(\frac{\Delta\lambda}{2}\right)
}.
$$

`sklearn.haversine_distances` oczekuje radianów i zwraca odległość kątową. Wynik mnożymy przez promień Ziemi.

### Pseudokod

```text
coords = kolumny lat, lon
coords_rad = zamień stopnie na radiany
D_city_km = haversine_distances(coords_rad) * R_Ziemi
```

### Pseudokod bliski Pythonowi

```python
city_coords_deg = cities[["lat", "lon"]].to_numpy()
city_coords_rad = np.radians(city_coords_deg)
D_city_km = haversine_distances(city_coords_rad) * R_EARTH_KM
```


In [ ]:
# TODO 1
city_coords_rad = None
D_city_km = None

In [ ]:
assert city_coords_rad is not None and D_city_km is not None
assert D_city_km.shape == (len(cities), len(cities))
assert np.allclose(np.diag(D_city_km), 0)
assert np.allclose(D_city_km, D_city_km.T)

print("Kształt macierzy:", D_city_km.shape)
print("Pamięć macierzy [MB]:", round(D_city_km.nbytes / 1024**2, 2))
print(
    "Mediana niezerowych odległości [km]:",
    round(np.median(D_city_km[D_city_km > 0]), 2)
)

plt.figure(figsize=(7, 7))
plt.scatter(cities["lon"], cities["lat"], s=10, alpha=0.75)
plt.xlabel("długość geograficzna")
plt.ylabel("szerokość geograficzna")
plt.title("Miasta w zbiorze PRNG")
plt.axis("equal")
plt.show()

## 3. `TODO 2` — binarna macierz sąsiedztwa $A$ i macierz podobieństwa $W$

Dla progu $\varepsilon$:

$$
A_{ij}=
\begin{cases}
1,&0<D_{ij}\leq\varepsilon,\\
0,&\text{inaczej}.
\end{cases}
$$

Następnie ważymy istniejące krawędzie:

$$
W_{ij}
=
A_{ij}
\exp\left(
-\frac{D_{ij}^2}{2\sigma^2}
\right).
$$

Interpretacja:

- `degree_binary` — liczba miast w promieniu $\varepsilon$,
- `degree_weighted` — łączna siła tych relacji, większa dla bardzo bliskich miast.

Dwa miasta mogą mieć tę samą liczbę sąsiadów, ale różny stopień ważony.

### Pseudokod bliski Pythonowi

```python
A_city = ((D_city_km <= EPS_CITY_GRAPH) & (D_city_km > 0)).astype(int)
W_city = A_city * np.exp(-(D_city_km ** 2) / (2 * SIGMA_CITY_KM ** 2))
np.fill_diagonal(W_city, 0)

degree_binary = A_city.sum(axis=1)
degree_weighted = W_city.sum(axis=1)
```


In [ ]:
EPS_CITY_GRAPH = 25.0
SIGMA_CITY_KM = 15.0

# TODO 2
A_city = None
W_city = None
degree_binary = None
degree_weighted = None

In [ ]:
assert A_city is not None and W_city is not None
assert np.all(np.diag(A_city) == 0)
assert np.allclose(A_city, A_city.T)
assert np.allclose(W_city, W_city.T)
assert np.all(W_city[A_city == 0] == 0)
assert np.all((W_city >= 0) & (W_city <= 1))

cities["degree_binary"] = degree_binary
cities["degree_weighted"] = degree_weighted

sample_ids = np.arange(min(12, len(cities)))
sample_names = cities.loc[sample_ids, "name"].tolist()

print("Fragment A_city")
display(pd.DataFrame(
    A_city[np.ix_(sample_ids, sample_ids)],
    index=sample_names,
    columns=sample_names
))

print("Fragment W_city")
display(pd.DataFrame(
    W_city[np.ix_(sample_ids, sample_ids)],
    index=sample_names,
    columns=sample_names
).round(3))

## 4. `TODO 3` — skan progu $\varepsilon$

Chcemy zbadać, jak wraz ze wzrostem promienia zmieniają się:

- liczba krawędzi,
- średni stopień,
- liczba składowych,
- udział miast w największej składowej.

To odpowiednik ćwiczenia z małej macierzy, ale teraz na realnym zbiorze 1020 miast.

### Pseudokod

```text
dla każdego eps:
    A = (0 < D <= eps)
    connected_components(A)
    policz rozmiary składowych
    zapisz statystyki
```

### Pseudokod bliski Pythonowi

```python
for eps in eps_grid_city:
    A = ((D_city_km <= eps) & (D_city_km > 0)).astype(np.int8)
    n_components, component_labels = connected_components(
        csr_matrix(A), directed=False
    )
    sizes = np.bincount(component_labels)
    rows.append({
        "eps_km": eps,
        "edges": int(A.sum() // 2),
        "mean_degree": float(A.sum(axis=1).mean()),
        "components": int(n_components),
        "largest_component_share": float(sizes.max() / len(A)),
    })
```


In [ ]:
eps_grid_city = np.arange(5.0, 55.5, 0.5)
rows = []

# TODO 3
for eps in eps_grid_city:
    # A = ...
    # n_components, labels = connected_components(...)
    # sizes = ...
    # rows.append({...})
    pass

city_scan = pd.DataFrame(rows)

In [ ]:
assert len(city_scan) == len(eps_grid_city), "Uzupełnij TODO 3."
assert city_scan["edges"].is_monotonic_increasing
assert city_scan["components"].is_monotonic_decreasing

city_scan["new_edges"] = city_scan["edges"].diff().fillna(city_scan["edges"])
city_scan["delta_giant_share"] = city_scan["largest_component_share"].diff().fillna(0)


def first_eps_reaching(column, threshold):
    reached = city_scan[city_scan[column] >= threshold]
    return float(reached.iloc[0]["eps_km"]) if len(reached) else np.nan

critical_thresholds = pd.DataFrame({
    "zdarzenie": [
        "największa składowa >= 50%",
        "największa składowa >= 90%",
        "największa składowa >= 99%",
        "graf spójny",
    ],
    "eps_km": [
        first_eps_reaching("largest_component_share", 0.50),
        first_eps_reaching("largest_component_share", 0.90),
        first_eps_reaching("largest_component_share", 0.99),
        float(city_scan.loc[city_scan["components"].eq(1), "eps_km"].iloc[0]),
    ],
})

display(critical_thresholds.round(2))
display(
    city_scan.loc[
        city_scan["eps_km"].isin([10, 15, 18, 20, 22, 25, 30, 40, 50])
    ].round(3)
)

# 1. Liczba wszystkich krawędzi — funkcja skumulowana odległości par miast.
plt.figure(figsize=(8, 4))
plt.plot(city_scan["eps_km"], city_scan["edges"])
plt.xlabel("eps [km]")
plt.ylabel("liczba krawędzi")
plt.title("Ile par miast mieści się w zadanym promieniu?")
plt.grid(alpha=0.25)
plt.show()

# 2. Ile nowych krawędzi pojawia się po zwiększeniu progu o 0.5 km?
plt.figure(figsize=(8, 4))
plt.plot(city_scan["eps_km"], city_scan["new_edges"])
plt.xlabel("eps [km]")
plt.ylabel("nowe krawędzie na krok 0.5 km")
plt.title("Przyrost liczby krawędzi")
plt.grid(alpha=0.25)
plt.show()

# 3. Sklejanie składowych.
plt.figure(figsize=(8, 4))
plt.plot(city_scan["eps_km"], city_scan["components"])
plt.xlabel("eps [km]")
plt.ylabel("liczba składowych")
plt.title("Sklejanie grafu miast")
plt.grid(alpha=0.25)
plt.show()

# 4. Najbardziej czytelne przejście strukturalne: rozmiar największej składowej.
plt.figure(figsize=(8, 4))
plt.plot(city_scan["eps_km"], city_scan["largest_component_share"])
for level in [0.50, 0.90, 0.99]:
    plt.axhline(level, linestyle="--", label=f"{int(level * 100)}% miast")
plt.xlabel("eps [km]")
plt.ylabel("udział największej składowej")
plt.title("Pojawienie się gigantycznej składowej")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

print("Największe pojedyncze skoki udziału największej składowej:")
display(
    city_scan.nlargest(8, "delta_giant_share")[[
        "eps_km", "delta_giant_share", "largest_component_share",
        "components", "edges",
    ]].round(3)
)

print(
    "Interpretacja: liczba krawędzi rośnie prawie zawsze, ale najciekawszy "
    "moment strukturalny widać na gwałtownym wzroście największej składowej."
)
# 5. Panel przejścia strukturalnego: krawędzie rosną płynnie,
#    lecz największa składowa może rosnąć skokowo.
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

axes[0].bar(
    city_scan['eps_km'],
    city_scan['new_edges'],
    width=0.42,
)
axes[0].set_ylabel('nowe krawędzie / 0.5 km')
axes[0].set_title('Nowe krawędzie pojawiają się stopniowo')
axes[0].grid(alpha=0.20)

axes[1].bar(
    city_scan['eps_km'],
    100 * city_scan['delta_giant_share'],
    width=0.42,
)
axes[1].set_xlabel('eps [km]')
axes[1].set_ylabel('skok największej składowej [p.p.]')
axes[1].set_title('Przejście strukturalne: największa składowa rośnie skokowo')
axes[1].grid(alpha=0.20)

plt.tight_layout()
plt.show()


## 5. `TODO 4` — graf $\varepsilon$ kontra graf $kNN$

Obie konstrukcje zaczynają od tej samej macierzy odległości $D$, ale odpowiadają na inne pytania.

### Graf $\varepsilon$

$$
A^\varepsilon_{ij}=1
\quad\Longleftrightarrow\quad
0<D_{ij}\leq\varepsilon.
$$

- wszystkie krawędzie są krótsze niż zadany próg,
- punkty w gęstych obszarach mają wysoki stopień,
- punkty peryferyjne mogą być izolowane.

### Graf $kNN$

$$
A^{kNN}_{ij}=1
\quad\Longleftrightarrow\quad
j\in N_k(i).
$$

- każdy punkt wybiera dokładnie $k$ sąsiadów w wersji kierunkowej,
- w rzadkim regionie wybrany sąsiad może być bardzo daleko,
- `union` ułatwia spójność, a `mutual` pozostawia tylko relacje obustronne.

### Pseudokod bliski Pythonowi

```python
K_CITY = 4
neighbors = np.argsort(D_city_km, axis=1)[:, 1:K_CITY + 1]
A_knn_dir = np.zeros_like(D_city_km, dtype=np.int8)
A_knn_dir[np.arange(len(cities))[:, None], neighbors] = 1
A_knn_union = ((A_knn_dir + A_knn_dir.T) > 0).astype(np.int8)
A_knn_mutual = ((A_knn_dir + A_knn_dir.T) == 2).astype(np.int8)
```


In [ ]:

K_CITY = 4

# TODO 4: uzupełnij trzy macierze zgodnie z pseudokodem.
neighbors_city_knn = None
A_city_knn_dir = None
A_city_knn_union = None
A_city_knn_mutual = None


In [ ]:

assert neighbors_city_knn is not None
assert A_city_knn_dir is not None
assert A_city_knn_union is not None
assert A_city_knn_mutual is not None
assert np.all(A_city_knn_dir.sum(axis=1) == K_CITY)
assert np.all(A_city_knn_union == A_city_knn_union.T)
assert np.all(A_city_knn_mutual == A_city_knn_mutual.T)


def graph_statistics_from_adjacency(A, D):
    n_components, component_labels = connected_components(csr_matrix(A), directed=False)
    sizes = np.bincount(component_labels)
    upper = np.triu_indices_from(A, k=1)
    edge_distances = D[upper][A[upper] > 0]
    return {
        "edges": int(A.sum() // 2),
        "components": int(n_components),
        "largest_component_share": sizes.max() / len(A),
        "mean_degree": A.sum(axis=1).mean(),
        "median_edge_km": np.median(edge_distances),
        "max_edge_km": edge_distances.max(),
    }

comparison_graphs = []
for name, adjacency in [
    (f"epsilon, eps={EPS_CITY_GRAPH:g} km", A_city),
    (f"kNN union, k={K_CITY}", A_city_knn_union),
    (f"mutual kNN, k={K_CITY}", A_city_knn_mutual),
]:
    comparison_graphs.append({
        "graf": name,
        **graph_statistics_from_adjacency(adjacency, D_city_km),
    })

graph_comparison_table = pd.DataFrame(comparison_graphs)
display(graph_comparison_table.round(3))

# Pełny skan kilku wartości k — kod jest gotowy, aby skupić się na interpretacji.
knn_scan_rows = []
for k in [2, 3, 4, 5, 6, 8, 10]:
    neighbors = np.argsort(D_city_km, axis=1)[:, 1:k + 1]
    A_dir = np.zeros_like(D_city_km, dtype=np.int8)
    A_dir[np.arange(len(cities))[:, None], neighbors] = 1
    for graph_type, adjacency in [
        ("union", ((A_dir + A_dir.T) > 0).astype(np.int8)),
        ("mutual", ((A_dir + A_dir.T) == 2).astype(np.int8)),
    ]:
        knn_scan_rows.append({
            "k": k,
            "typ": graph_type,
            **graph_statistics_from_adjacency(adjacency, D_city_km),
        })

knn_scan = pd.DataFrame(knn_scan_rows)
display(knn_scan.round(3))

plt.figure(figsize=(8, 4))
for graph_type, group in knn_scan.groupby("typ"):
    plt.plot(group["k"], group["largest_component_share"], marker="o", label=graph_type)
plt.xlabel("k")
plt.ylabel("udział największej składowej")
plt.title("Spójność grafu kNN zależnie od k i symetryzacji")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
for graph_type, group in knn_scan.groupby("typ"):
    plt.plot(group["k"], group["max_edge_km"], marker="o", label=graph_type)
plt.xlabel("k")
plt.ylabel("najdłuższa krawędź [km]")
plt.title("Cena spójności grafu kNN: długie krawędzie w rzadkich regionach")
plt.grid(alpha=0.25)
plt.legend()
plt.show()
# Porównanie pełnych rozkładów długości krawędzi.
def edge_lengths_from_adjacency(A, D):
    upper = np.triu_indices_from(A, k=1)
    return D[upper][A[upper] > 0]

edge_length_sets = {
    f'epsilon {EPS_CITY_GRAPH:g} km': edge_lengths_from_adjacency(A_city, D_city_km),
    f'kNN union k={K_CITY}': edge_lengths_from_adjacency(A_city_knn_union, D_city_km),
    f'mutual kNN k={K_CITY}': edge_lengths_from_adjacency(A_city_knn_mutual, D_city_km),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for label, values in edge_length_sets.items():
    axes[0].hist(values, bins=35, alpha=0.45, label=label)
    ordered = np.sort(values)
    ecdf = np.arange(1, len(ordered) + 1) / len(ordered)
    axes[1].plot(ordered, ecdf, label=label)

axes[0].set_xlabel('długość krawędzi [km]')
axes[0].set_ylabel('liczba krawędzi')
axes[0].set_title('Rozkład długości krawędzi')
axes[0].legend()
axes[0].grid(alpha=0.20)

axes[1].set_xlabel('długość krawędzi [km]')
axes[1].set_ylabel('udział krawędzi krótszych niż x')
axes[1].set_title('Dystrybuanta empiryczna długości')
axes[1].legend()
axes[1].grid(alpha=0.20)

plt.tight_layout()
plt.show()

## 6. `TODO 5` — huby binarne i ważone

W grafie binarnym hub ma wielu sąsiadów.  
W grafie ważonym hub ma dużą **łączną siłę** bliskich relacji.

Porównaj rankingi:

- czy te same miasta są wysoko w obu?
- czy stopień ważony premiuje zwarte konurbacje?
- czego ta miara **nie** mówi? — nie mówi bezpośrednio o populacji ani liczbie podróży.

### Pseudokod bliski Pythonowi

```python
columns = ["name", "wojewodztwo", "degree_binary", "degree_weighted"]
hubs_binary = cities.nlargest(15, "degree_binary")[columns]
hubs_weighted = cities.nlargest(15, "degree_weighted")[columns]
```


In [ ]:
# TODO 5
hubs_binary = None
hubs_weighted = None

In [ ]:
assert hubs_binary is not None and hubs_weighted is not None

print("Top huby według stopnia binarnego")
display(hubs_binary.round(3))

print("Top huby według stopnia ważonego")
display(hubs_weighted.round(3))

plt.figure(figsize=(7, 7))
sc = plt.scatter(
    cities["lon"], cities["lat"],
    c=cities["degree_weighted"], s=13
)
plt.colorbar(sc, label="stopień ważony")
plt.xlabel("lon")
plt.ylabel("lat")
plt.title(
    f"Huby ważone: eps={EPS_CITY_GRAPH:g} km, sigma={SIGMA_CITY_KM:g} km"
)
plt.axis("equal")
plt.show()

print(
    "Korelacja stopnia binarnego i ważonego:",
    round(cities[["degree_binary", "degree_weighted"]].corr(method="spearman").iloc[0, 1], 3)
)


### Dlaczego mapa całej Polski jest mało czytelna?

Na mapie kraju większość lokalnych krawędzi jest bardzo krótka w stosunku do całego obszaru. Dlatego do interpretacji grafu lepszy jest **zoom regionalny**. Porównamy dwa kontrasty:

- `śląskie` — gęsta sieć miejska,
- `podlaskie` — rzadsza sieć i dłuższe relacje kNN.


In [ ]:

def plot_city_graph_region(province, adjacency, title, max_edges=900):
    ids = np.flatnonzero(cities["wojewodztwo"].eq(province).to_numpy())
    local_A = adjacency[np.ix_(ids, ids)]
    edge_i, edge_j = np.where(np.triu(local_A, k=1) > 0)

    if len(edge_i) > max_edges:
        keep = np.linspace(0, len(edge_i) - 1, max_edges, dtype=int)
        edge_i, edge_j = edge_i[keep], edge_j[keep]

    plt.figure(figsize=(7, 7))
    for a, b in zip(edge_i, edge_j):
        i, j = ids[a], ids[b]
        plt.plot(
            [cities.loc[i, "lon"], cities.loc[j, "lon"]],
            [cities.loc[i, "lat"], cities.loc[j, "lat"]],
            linewidth=0.7, alpha=0.35,
        )
    plt.scatter(cities.loc[ids, "lon"], cities.loc[ids, "lat"], s=25)
    plt.xlabel("lon")
    plt.ylabel("lat")
    plt.title(title)
    plt.axis("equal")
    plt.show()

plot_city_graph_region(
    "śląskie", A_city,
    f"Śląskie — graf epsilon, eps={EPS_CITY_GRAPH:g} km",
)
plot_city_graph_region(
    "podlaskie", A_city_knn_union,
    f"Podlaskie — graf kNN union, k={K_CITY}",
)
# Ten sam region dla czterech wartości eps — wyraźny efekt sklejania.
def draw_region_graph_on_axis(province, adjacency, ax, title, max_edges=1100):
    ids = np.flatnonzero(cities['wojewodztwo'].eq(province).to_numpy())
    local_A = adjacency[np.ix_(ids, ids)]
    edge_i, edge_j = np.where(np.triu(local_A, k=1) > 0)

    if len(edge_i) > max_edges:
        keep = np.linspace(0, len(edge_i) - 1, max_edges, dtype=int)
        edge_i, edge_j = edge_i[keep], edge_j[keep]

    for a, b in zip(edge_i, edge_j):
        i, j = ids[a], ids[b]
        ax.plot(
            [cities.loc[i, 'lon'], cities.loc[j, 'lon']],
            [cities.loc[i, 'lat'], cities.loc[j, 'lat']],
            linewidth=0.65, alpha=0.32,
        )
    ax.scatter(cities.loc[ids, 'lon'], cities.loc[ids, 'lat'], s=20)
    ax.set_title(title)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(alpha=0.20)

eps_silesia = [15, 20, 25, 30]
fig, axes = plt.subplots(2, 2, figsize=(12, 11))
for ax, eps_local in zip(axes.ravel(), eps_silesia):
    A_local_eps = (
        (D_city_km <= eps_local) & (D_city_km > 0)
    ).astype(np.int8)
    draw_region_graph_on_axis(
        'śląskie',
        A_local_eps,
        ax,
        f'Śląskie — eps={eps_local} km',
    )
plt.suptitle('Śląskie — wzrost grafu epsilon i sklejanie lokalnej sieci')
plt.tight_layout()
plt.show()

## 7. `TODO 6` — DBSCAN jako rozszerzenie grafu $\varepsilon$

DBSCAN przy `metric="precomputed"` otrzymuje tę samą macierz $D_{\text{city}}$.

Dla danego `eps`:

- otoczenia są równoważne wierszom macierzy $A_\varepsilon$,
- `min_samples` określa minimalną lokalną gęstość,
- algorytm łączy osiągalne punkty `core`,
- izolowane miasta mogą zostać oznaczone jako szum.

### Pseudokod

```text
dla eps w liście:
    labels = DBSCAN(eps, min_samples, metric="precomputed").fit_predict(D)
    policz klastry bez etykiety -1
    policz udział szumu
```

### Pseudokod bliski Pythonowi

```python
for eps in eps_values:
    labels = DBSCAN(
        eps=eps,
        min_samples=3,
        metric="precomputed",
    ).fit_predict(D_city_km)
    n_clusters = len(set(labels) - {-1})
    noise_share = np.mean(labels == -1)

city_db_labels = DBSCAN(
    eps=EPS_DBSCAN,
    min_samples=3,
    metric="precomputed",
).fit_predict(D_city_km)
```


In [ ]:
db_rows = []

# TODO 6
for eps in [12, 15, 18, 20, 22, 25, 30]:
    # labels = ...
    # db_rows.append({...})
    pass

dbscan_scan = pd.DataFrame(db_rows)

EPS_DBSCAN = 20
city_db_labels = None

In [ ]:
assert len(dbscan_scan) > 0 and city_db_labels is not None
display(dbscan_scan.round(3))

cities["cluster_dbscan"] = city_db_labels

plt.figure(figsize=(7, 7))
plt.scatter(
    cities["lon"], cities["lat"],
    c=cities["cluster_dbscan"], s=12
)
plt.xlabel("lon")
plt.ylabel("lat")
plt.title(f"DBSCAN miast, eps={EPS_DBSCAN} km")
plt.axis("equal")
plt.show()

## Dodatkowe — Isomap miast Polski: mapa odległości czy mapa połączeń?

Geograficzna mapa $(lon,lat)$ pokazuje fizyczne położenie miast. Isomap odpowiada na inne pytanie:

> jak ułożyć miasta, aby odtworzyć odległości **po lokalnym grafie $kNN$**?

Schemat jest taki sam jak w notebooku 02:

$$
D_{city}
\longrightarrow A_{kNN}
\longrightarrow D^G
\longrightarrow \text{MDS}
\longrightarrow Y_{Isomap}.
$$

Wykres Isomap nie powinien być interpretowany jak kartograficzna mapa Polski. Pokazuje topologię lokalnych relacji. Miasta połączone krótkimi ścieżkami grafowymi powinny leżeć blisko w embeddingu.

Parametr `K_ISOMAP_CITY` kontroluje kompromis:

- zbyt małe $k$ — ryzyko rozspójnienia grafu,
- zbyt duże $k$ — długie skróty niszczą lokalną geometrię.

In [ ]:
K_ISOMAP_CITY = 4

isomap_city_model = Isomap(
    n_neighbors=K_ISOMAP_CITY,
    n_components=2,
    metric='precomputed',
)
Y_city_isomap = isomap_city_model.fit_transform(D_city_km)
D_city_graph = isomap_city_model.dist_matrix_
D_city_isomap_low = pairwise_distances(Y_city_isomap)

tri_city = np.triu_indices_from(D_city_km, k=1)
isomap_city_metrics = pd.Series({
    'trustworthiness względem D_haversine': trustworthiness(
        D_city_km,
        Y_city_isomap,
        n_neighbors=10,
        metric='precomputed',
    ),
    'Spearman(D_haversine, D_Y)': spearmanr(
        D_city_km[tri_city],
        D_city_isomap_low[tri_city],
    ).statistic,
    'Spearman(D_graph, D_Y)': spearmanr(
        D_city_graph[tri_city],
        D_city_isomap_low[tri_city],
    ).statistic,
})
display(isomap_city_metrics.round(3))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].scatter(
    cities['lon'], cities['lat'],
    c=cities['lon'], s=11,
)
axes[0].set_title('Położenie geograficzne; kolor = długość geograficzna')
axes[0].set_aspect('equal', adjustable='box')

axes[1].scatter(
    Y_city_isomap[:, 0], Y_city_isomap[:, 1],
    c=cities['lon'], s=11,
)
axes[1].set_title(f'Isomap miast, k={K_ISOMAP_CITY}; ten sam kolor')

axes[2].scatter(
    Y_city_isomap[:, 0], Y_city_isomap[:, 1],
    c=city_db_labels, s=11,
)
axes[2].set_title('Isomap; kolor = klaster DBSCAN')

for ax in axes:
    ax.grid(alpha=0.20)
plt.tight_layout()
plt.show()

### Interpretacja

- Wysoka korelacja $D^G$ z $D^Y$ oznacza, że Isomap rzeczywiście odtwarza odległości po grafie.
- Różnica pomiędzy $D_{haversine}$ i $D^G$ pokazuje wpływ lokalnej sieci sąsiedztwa.
- Jeżeli $kNN$ zawiera bardzo długie krawędzie w regionach peryferyjnych, mogą one działać jak skróty i zmieniać embedding.
- Isomap nie wskazuje automatycznie tras transportowych. Jest narzędziem do wizualizacji topologii grafu.

**Ćwiczenie:** zmień `K_ISOMAP_CITY` na 3, 5 i 8. Porównaj spójność, kształt embeddingu i korelację z $D^G$.

## 8. Dodatkowe — minimalne drzewo rozpinające

MST łączy wszystkie miasta przy minimalnej łącznej długości krawędzi. Nie jest propozycją sieci transportowej, ale może służyć jako geometryczny „kręgosłup” i wskazać największe luki potrzebne do utrzymania spójności.

Najdłuższa krawędź MST odpowiada najmniejszemu progowi $\varepsilon$, przy którym pełny graf progowy może stać się spójny.

In [ ]:
mst = minimum_spanning_tree(D_city_km).tocoo()
mst_edges = pd.DataFrame({
    "i": mst.row,
    "j": mst.col,
    "distance_km": mst.data,
})
mst_edges["city_i"] = cities.loc[mst_edges["i"], "name"].to_numpy()
mst_edges["city_j"] = cities.loc[mst_edges["j"], "name"].to_numpy()

display(
    mst_edges.sort_values("distance_km", ascending=False)
    .head(15)[["city_i", "city_j", "distance_km"]]
    .round(2)
)


## 9. Automatyczne podsumowanie części grafowej

Ta komórka nie zastępuje interpretacji studenta. Zbiera jednak najważniejsze liczby, aby łatwo przejść od wykresów do zdań analitycznych.


In [ ]:

eps50 = first_eps_reaching("largest_component_share", 0.50)
eps90 = first_eps_reaching("largest_component_share", 0.90)
eps99 = first_eps_reaching("largest_component_share", 0.99)
eps_connected = float(city_scan.loc[city_scan["components"].eq(1), "eps_km"].iloc[0])

largest_jump_row = city_scan.loc[city_scan["delta_giant_share"].idxmax()]
top_binary_names = ", ".join(cities.nlargest(5, "degree_binary")["name"])
top_weighted_names = ", ".join(cities.nlargest(5, "degree_weighted")["name"])

db20 = dbscan_scan.iloc[(dbscan_scan["eps_km"] - 20).abs().argsort()[:1]].iloc[0]

print(f"1. Największa składowa przekracza 50% przy około {eps50:.1f} km.")
print(f"2. Przekracza 90% przy około {eps90:.1f} km i 99% przy około {eps99:.1f} km.")
print(f"3. Pełna spójność pojawia się dopiero przy około {eps_connected:.1f} km.")
print(
    f"4. Największy pojedynczy skok największej składowej występuje przy eps≈"
    f"{largest_jump_row['eps_km']:.1f} km i wynosi "
    f"{100 * largest_jump_row['delta_giant_share']:.1f} punktów procentowych."
)
print(f"5. Top stopień binarny: {top_binary_names}.")
print(f"6. Top stopień ważony: {top_weighted_names}.")
print(
    f"7. DBSCAN dla eps≈20 km daje około {int(db20['clusters'])} klastrów "
    f"i {100 * db20['noise_share']:.1f}% szumu."
)
print(
    "8. Wniosek: przed pełną spójnością powstaje gigantyczna składowa; "
    "pozostałe komponenty to miasta peryferyjne wymagające dłuższych krawędzi."
)


# Część B — screening przestrzenny połączeń lokalnych

## 9. `TODO 7` — każda miejscowość do najbliższego miasta

Traktujemy miasta jako uproszczone centra usług. Dla każdej pozostałej miejscowości szukamy najbliższego miasta.

To nadal analiza geometryczna:

- odległość jest w linii prostej,
- nie znamy dróg ani czasu przejazdu,
- nie znamy liczby mieszkańców,
- nie znamy istniejących linii autobusowych.

### Pseudokod

```text
zbuduj BallTree na współrzędnych miast w radianach
zamień współrzędne pozostałych miejscowości na radiany
query(k=1)
odległość kątową pomnóż przez promień Ziemi
przepisz nazwę i współrzędne wybranego miasta
```

### Pseudokod bliski Pythonowi

```python
city_tree = BallTree(city_coords_rad, metric="haversine")
noncity_coords_rad = np.radians(noncities[["lat", "lon"]].to_numpy())
nearest_dist_rad, nearest_idx = city_tree.query(noncity_coords_rad, k=1)
nearest_idx = nearest_idx[:, 0]
```


In [ ]:
# TODO 7
city_tree = None
noncity_coords_rad = None
nearest_dist_rad = None
nearest_idx = None

In [ ]:
assert city_tree is not None and nearest_idx is not None
assert len(nearest_idx) == len(noncities)

noncities["nearest_city_dist_km"] = (
    nearest_dist_rad[:, 0] * R_EARTH_KM
)
noncities["nearest_city_id"] = cities.iloc[nearest_idx]["id"].to_numpy()
noncities["nearest_city_name"] = cities.iloc[nearest_idx]["name"].to_numpy()
noncities["nearest_city_gmina"] = cities.iloc[nearest_idx]["gmina"].to_numpy()
noncities["nearest_city_powiat"] = cities.iloc[nearest_idx]["powiat"].to_numpy()
noncities["nearest_city_wojewodztwo"] = cities.iloc[nearest_idx]["wojewodztwo"].to_numpy()
noncities["nearest_city_lat"] = cities.iloc[nearest_idx]["lat"].to_numpy()
noncities["nearest_city_lon"] = cities.iloc[nearest_idx]["lon"].to_numpy()

print(
    noncities["nearest_city_dist_km"]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .round(2)
)

plt.figure(figsize=(7.5, 4))
plt.hist(noncities["nearest_city_dist_km"], bins=60)
plt.axvline(15, linestyle="--", label="15 km")
plt.axvline(25, linestyle=":", label="25 km")
plt.xlabel("odległość do najbliższego miasta [km]")
plt.ylabel("liczba miejscowości")
plt.title("Geometryczna dostępność do najbliższego miasta")
plt.legend()
plt.show()

## 10. `TODO 8` — agregacja do gmin i wskaźnik screeningowy

Dla każdej gminy liczymy między innymi:

- liczbę miejscowości niebędących miastami,
- średnią, medianę, 90. percentyl i maksimum odległości,
- udział miejscowości dalej niż 15 i 25 km,
- sumę odległości jako prosty „burden”.

Następnie łączymy rangi percentylowe w indeks:

$$
\text{need score}
=
100\sum_m \alpha_m\,\operatorname{rank}_{pct}(z_m),
\qquad
\sum_m\alpha_m=1.
$$

To konstrukcja analityczna, a nie obserwowany popyt.

### Pseudokod bliski Pythonowi

```python
noncities["gt15"] = noncities["nearest_city_dist_km"] > 15
noncities["gt25"] = noncities["nearest_city_dist_km"] > 25

gmina_need = (
    noncities.groupby(
        ["gmina_key", "wojewodztwo", "powiat", "gmina"],
        as_index=False,
    )
    .agg(
        n_settlements=("name", "size"),
        mean_km=("nearest_city_dist_km", "mean"),
        median_km=("nearest_city_dist_km", "median"),
        p90_km=("nearest_city_dist_km", lambda s: s.quantile(0.90)),
        max_km=("nearest_city_dist_km", "max"),
        share_gt15=("gt15", "mean"),
        share_gt25=("gt25", "mean"),
        burden_km=("nearest_city_dist_km", "sum"),
        lat_centroid=("lat", "mean"),
        lon_centroid=("lon", "mean"),
    )
)
```


In [ ]:
# TODO 8: uzupełnij agregację.
gmina_need = None

In [ ]:
assert gmina_need is not None and len(gmina_need) > 0

weights = {
    "burden_km": 0.30,
    "mean_km": 0.25,
    "share_gt15": 0.20,
    "max_km": 0.15,
    "n_settlements": 0.10,
}
assert np.isclose(sum(weights.values()), 1.0)

gmina_need["need_score"] = 100 * sum(
    weight * gmina_need[column].rank(pct=True)
    for column, weight in weights.items()
)

gmina_need["priority_class"] = pd.qcut(
    gmina_need["need_score"],
    q=[0, 0.50, 0.75, 0.90, 1.0],
    labels=["niski", "umiarkowany", "wysoki", "bardzo wysoki"],
)

display(
    gmina_need.sort_values("need_score", ascending=False)
    .head(20)
    .round(2)
)

## 11. Analiza regionalna

Wybierz województwo. Dobrymi kontrastami są:

- `podlaskie` — rozproszona i peryferyjna struktura,
- `warmińsko-mazurskie`,
- `lubelskie`,
- `śląskie` — gęsta sieć miejska.

Zapisz:

1. trzy gminy o najwyższym `need_score`,
2. co powoduje ich wysoką pozycję,
3. czy wynik jest zgodny z mapą,
4. jakich danych brakuje do decyzji transportowej.

In [ ]:
SELECT_PROVINCE = "podlaskie"  # TODO: wybierz województwo i opisz wynik

region_need = gmina_need[
    gmina_need["wojewodztwo"].eq(SELECT_PROVINCE)
].copy()

display(
    region_need.sort_values("need_score", ascending=False)
    .head(15)
    .round(2)
)

plt.figure(figsize=(7, 7))
sc = plt.scatter(
    region_need["lon_centroid"],
    region_need["lat_centroid"],
    c=region_need["need_score"],
    s=38
)
plt.colorbar(sc, label="need_score 0–100")
plt.xlabel("lon")
plt.ylabel("lat")
plt.title(f"Screening przestrzenny — {SELECT_PROVINCE}")
plt.axis("equal")
plt.show()

## 12. `TODO 9` — kandydackie relacje gmina $\rightarrow$ miasto

Agregujemy miejscowości według:

- gminy źródłowej,
- najbliższego miasta.

Prosty priorytet relacji:

$$
\text{route priority}
=
n_{\text{miejscowości}}
\cdot
\overline d
\cdot
(1+\text{udział}_{>15\text{ km}}).
$$

Wysoka wartość oznacza, że:

- wiele punktów osadniczych ciąży do tego samego miasta,
- średnia odległość jest duża,
- dużo punktów leży dalej niż 15 km.

To nadal ranking do **weryfikacji**, nie automatyczna rekomendacja uruchomienia linii.

### Pseudokod bliski Pythonowi

```python
relations = (
    noncities.groupby(
        [
            "gmina_key", "wojewodztwo", "powiat", "gmina",
            "nearest_city_id", "nearest_city_name", "nearest_city_gmina",
            "nearest_city_lat", "nearest_city_lon",
        ],
        as_index=False,
    )
    .agg(
        n_settlements=("name", "size"),
        mean_km=("nearest_city_dist_km", "mean"),
        max_km=("nearest_city_dist_km", "max"),
        share_gt15=("gt15", "mean"),
        source_lat=("lat", "mean"),
        source_lon=("lon", "mean"),
        burden_km=("nearest_city_dist_km", "sum"),
    )
)
```


In [ ]:
# TODO 9
relations = None

In [ ]:
assert relations is not None and len(relations) > 0

relations["cross_gmina"] = (
    relations["gmina"] != relations["nearest_city_gmina"]
)
relations["route_priority"] = (
    relations["n_settlements"]
    * relations["mean_km"]
    * (1 + relations["share_gt15"])
)

region_relations = relations[
    relations["wojewodztwo"].eq(SELECT_PROVINCE)
    & relations["cross_gmina"]
].sort_values("route_priority", ascending=False)

display(
    region_relations.head(15)[[
        "gmina", "nearest_city_name",
        "n_settlements", "mean_km", "max_km",
        "share_gt15", "route_priority"
    ]].round(2)
)

TOP_ROUTES_TO_DRAW = min(15, len(region_relations))
top_routes = region_relations.head(TOP_ROUTES_TO_DRAW)

plt.figure(figsize=(8, 8))
plt.scatter(
    region_need["lon_centroid"],
    region_need["lat_centroid"],
    s=18, alpha=0.45, label="centroidy gmin"
)
for _, row in top_routes.iterrows():
    plt.plot(
        [row["source_lon"], row["nearest_city_lon"]],
        [row["source_lat"], row["nearest_city_lat"]],
        linewidth=1.4, alpha=0.75
    )
    plt.scatter(
        row["nearest_city_lon"],
        row["nearest_city_lat"],
        marker="x", s=50
    )
plt.xlabel("lon")
plt.ylabel("lat")
plt.title(
    f"Top {TOP_ROUTES_TO_DRAW} relacji do weryfikacji — {SELECT_PROVINCE}"
)
plt.axis("equal")
plt.legend()
plt.show()

## 13. `TODO 10` — scenariusz koncentracji rankingu

Sprawdzamy, jaką część:

- miejscowości,
- łącznego `burden_km`

obejmuje pierwszych $k$ relacji.

To odpowiada na pytanie:

> Czy niewielka liczba relacji obejmuje dużą część problemu screeningowego, czy potrzeby są rozproszone?

### Pseudokod bliski Pythonowi

```python
scenario = region_relations.copy().reset_index(drop=True)
scenario["rank"] = np.arange(1, len(scenario) + 1)
scenario["cum_settlements"] = scenario["n_settlements"].cumsum()
scenario["cum_burden"] = scenario["burden_km"].cumsum()
scenario["coverage_share"] = scenario["cum_settlements"] / scenario["n_settlements"].sum()
scenario["cumulative_burden_share"] = scenario["cum_burden"] / scenario["burden_km"].sum()
```


In [ ]:
# TODO 10
scenario = None

In [ ]:
assert scenario is not None and len(scenario) > 0

show_k = min(100, len(scenario))
plt.figure(figsize=(7.5, 4))
plt.plot(
    scenario["rank"].head(show_k),
    scenario["coverage_share"].head(show_k),
    label="udział miejscowości"
)
plt.plot(
    scenario["rank"].head(show_k),
    scenario["cumulative_burden_share"].head(show_k),
    label="udział burden"
)
plt.xlabel("liczba najwyżej ocenionych relacji")
plt.ylabel("udział skumulowany")
plt.title(f"Koncentracja rankingu — {SELECT_PROVINCE}")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

checkpoints = [1, 5, 10, 20, 50, 100]
display(
    scenario.loc[
        scenario["rank"].isin(checkpoints),
        ["rank", "coverage_share", "cumulative_burden_share"]
    ].round(3)
)

## 14. Dodatkowe — drzewo jako opis skonstruowanego wskaźnika

Definiujemy klasę:

$$
\text{high need}=1
$$

dla gmin z górnego kwartylu `need_score`.

Ponieważ `need_score` został skonstruowany z części tych samych cech, drzewo jest tutaj **modelem surrogate** — ma uprościć i opisać reguły wskaźnika. Nie jest niezależnym modelem popytu.

To dobry przykład różnicy pomiędzy:

- interpretacją reguł własnego wskaźnika,
- prawdziwą predykcją zewnętrznego targetu, np. liczby pasażerów.

### Pseudokod bliski Pythonowi

```python
features = ["n_settlements", "mean_km", "max_km", "share_gt15", "share_gt25"]
target = (gmina_need["need_score"] >= gmina_need["need_score"].quantile(0.75)).astype(int)

X_train_tree, X_test_tree, y_train_tree, y_test_tree = train_test_split(
    gmina_need[features], target,
    test_size=0.25,
    stratify=target,
    random_state=RANDOM_STATE,
)

tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=25, random_state=RANDOM_STATE)
tree.fit(X_train_tree, y_train_tree)
```


### Co konkretnie daje drzewo?

Drzewo zamienia wielowymiarowy wskaźnik na zestaw prostych reguł progowych, np.:

```text
jeżeli średnia odległość > próg
    i udział miejscowości dalej niż 15 km > próg,
to wysoki priorytet screeningu.
```

Dzięki temu można:

- zobaczyć, które zmienne dominują,
- wykryć interakcje i progi,
- wytłumaczyć ranking osobie nietechnicznej,
- znaleźć gminy odstające od prostych reguł.

Nie wolno jednak interpretować drzewa jako dowodu przyczynowego. W tej sekcji target pochodzi z naszego `need_score`, więc drzewo **streszcza skonstruowany wskaźnik**, a nie odkrywa rzeczywistego popytu.


In [ ]:
# TODO 11 — opcjonalne:
# 1. wybierz cechy,
# 2. zbuduj target jako górny kwartyl need_score,
# 3. wykonaj podział train/test,
# 4. dopasuj DecisionTreeClassifier.
features = None
tree = None
X_test_tree = None
y_test_tree = None

In [ ]:
if tree is None:
    print("Sekcja opcjonalna: uzupełnij TODO 10 albo przejdź do wniosków.")
else:
    pred_tree = tree.predict(X_test_tree)
    proba_tree = tree.predict_proba(X_test_tree)[:, 1]

    print("ROC AUC surrogate:", round(roc_auc_score(y_test_tree, proba_tree), 3))
    print(classification_report(y_test_tree, pred_tree, digits=3))
    print(export_text(tree, feature_names=features))

    plt.figure(figsize=(16, 7))
    plot_tree(
        tree,
        feature_names=features,
        class_names=["pozostałe", "high_need"],
        filled=False,
        fontsize=8
    )
    plt.title("Drzewo jako uproszczony opis skonstruowanego indeksu")
    plt.show()

## 14B. Dwa znaczenia drzewa: streszczenie wskaźnika a próba generalizacji

Warto rozdzielić dwa eksperymenty.

### Drzewo A — diagnostyczne

Otrzymuje składniki, z których zbudowano `need_score`, np. `mean_km`, `max_km`, `share_gt15`. Powinno osiągać bardzo dobry wynik, ponieważ odtwarza prawie ten sam wzór w postaci reguł.

To nie jest przeciek przez pomyłkę — może być świadomym **modelem surrogate**, który tłumaczy złożony score prostymi progami.

### Drzewo B — kontekstowe

Nie otrzymuje bezpośrednich składników score. Korzysta z szerszych cech przestrzennych PRNG, np.:

- położenia centroidu,
- liczby i udziału miast/wsi,
- rozproszenia miejscowości,
- odległości do innych gmin,
- stopni w grafie gmin.

To trudniejsze i ciekawsze pytanie:

> czy ogólna struktura przestrzenna pozwala przewidzieć, które gminy trafią do wysokiego priorytetu?

Jeśli Drzewo A jest znacznie lepsze od B, nie oznacza to, że A jest „mądrzejsze”. A zna konstrukcję targetu; B próbuje generalizować z kontekstu.

In [ ]:
spatial_path_for_tree = locate_data('prng_gmina_spatial_features_v6.csv')
spatial_for_tree = pd.read_csv(spatial_path_for_tree)

tree_data = gmina_need.merge(
    spatial_for_tree,
    on='gmina_key',
    how='inner',
    suffixes=('', '_spatial'),
)
tree_data['high_need'] = (
    tree_data['need_score'] >= tree_data['need_score'].quantile(0.75)
).astype(int)

direct_features = [
    'n_settlements', 'mean_km', 'max_km',
    'share_gt15', 'lat_centroid', 'lon_centroid',
]
context_features = [
    'lat_centroid_prng', 'lon_centroid_prng',
    'n_places_prng', 'city_share_prng',
    'mean_dist_to_centroid_km_prng',
    'max_dist_to_centroid_km_prng',
    'nearest_gmina_km', 'mean_5_nearest_gminas_km',
    'gmina_centroid_degree_eps_25km',
    'gmina_centroid_degree_eps_50km',
]

train_ids, test_ids = train_test_split(
    np.arange(len(tree_data)),
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=tree_data['high_need'],
)

tree_comparison_rows = []
fitted_tree_models = {}

for experiment_name, feature_names in {
    'A: składniki need_score (surrogate)': direct_features,
    'B: szerszy kontekst przestrzenny': context_features,
}.items():
    X_tree_all = tree_data[feature_names].copy()
    X_tree_all = X_tree_all.fillna(X_tree_all.median(numeric_only=True))
    y_tree_all = tree_data['high_need']

    model = DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=25,
        random_state=RANDOM_STATE,
    )
    model.fit(X_tree_all.iloc[train_ids], y_tree_all.iloc[train_ids])

    predicted = model.predict(X_tree_all.iloc[test_ids])
    probability = model.predict_proba(X_tree_all.iloc[test_ids])[:, 1]

    tree_comparison_rows.append({
        'eksperyment': experiment_name,
        'accuracy': accuracy_score(y_tree_all.iloc[test_ids], predicted),
        'ROC_AUC': roc_auc_score(y_tree_all.iloc[test_ids], probability),
        'liczba_cech': len(feature_names),
    })
    fitted_tree_models[experiment_name] = (model, feature_names)

tree_comparison = pd.DataFrame(tree_comparison_rows)
display(tree_comparison.round(3))

context_name = 'B: szerszy kontekst przestrzenny'
context_tree, context_feature_names = fitted_tree_models[context_name]

plt.figure(figsize=(18, 8))
plot_tree(
    context_tree,
    feature_names=context_feature_names,
    class_names=['pozostałe', 'high need'],
    filled=False,
    fontsize=8,
)
plt.title('Drzewo B — próba wyjaśnienia high need bez bezpośrednich składników score')
plt.show()

### Jak interpretować wynik?

- Bardzo wysoki wynik Drzewa A jest oczekiwany: target został zbudowany z tych samych zmiennych.
- Wynik Drzewa B mówi, ile informacji o rankingu zawiera sama struktura przestrzenna.
- Żadne z drzew nie prognozuje jeszcze rzeczywistego popytu transportowego.
- Do modelu predykcyjnego potrzebny jest niezależny target, np. liczba pasażerów, frekwencja kursów, liczba dojazdów lub luka w pokryciu GTFS.


# 15. Dodatkowe — analiza wielowymiarowa gmin: PCA, t-SNE, UMAP i modele drzewiaste

Dotychczas analizowaliśmy głównie geometrię. Możemy jednak opisać każdą gminę wieloma zmiennymi i utworzyć macierz:

$$
X_{gmina}
=
[\text{cechy osadnicze},\ \text{dostępność},\ \text{demografia},\ \text{gospodarka},\ \text{edukacja},\ldots].
$$

Do paczki dołączono realne cechy przestrzenne policzone z PRNG. Opcjonalnie można dołączyć z BDL/SMUP/CKE m.in.:

- ludność i gęstość zaludnienia,
- saldo migracji i strukturę wieku,
- dochody własne oraz wydatki inwestycyjne per capita,
- liczbę podmiotów REGON na 10 tys. mieszkańców,
- bezrobocie lub wynagrodzenia — często dostępne na poziomie powiatu,
- wyniki edukacyjne,
- liczbę szkół, przychodni, przystanków i dostęp do kolei.

Po standaryzacji można porównać:

- **PCA** — liniowe kierunki zmienności,
- **t-SNE** — lokalne podobieństwo gmin jako rozkład $P$,
- **UMAP** — lokalny fuzzy graf $kNN$,
- **k-means/DBSCAN** — typologię gmin,
- **Decision Tree/Random Forest/XGBoost** — interpretację lub predykcję zewnętrznego targetu.

Najciekawszy eksperyment:

> porównać model tylko na cechach społeczno-ekonomicznych z modelem, który dodatkowo otrzymuje cechy grafowe. Czy graf wnosi informację ponad zwykłą tabelę?

Sekcja jest domyślnie wyłączona, aby zmieścić część obowiązkową w 3 godzinach.


In [ ]:

RUN_MULTIDIMENSIONAL_EXTENSION = False

if RUN_MULTIDIMENSIONAL_EXTENSION:
    spatial_path = locate_data("prng_gmina_spatial_features_v6.csv")
    spatial = pd.read_csv(spatial_path)

    feature_candidates = [
        "n_places_prng",
        "city_share_prng",
        "mean_nn_place_km_within_gmina_prng",
        "mean_dist_to_centroid_km_prng",
        "max_dist_to_centroid_km_prng",
        "nearest_gmina_km",
        "mean_5_nearest_gminas_km",
        "weighted_degree_exp_scale_25km",
        "weighted_degree_exp_scale_50km",
    ]

    feature_frame = spatial[feature_candidates].copy()
    feature_frame = feature_frame.fillna(feature_frame.median(numeric_only=True))
    X_multi = StandardScaler().fit_transform(feature_frame)

    Y_pca_gmina = PCA(n_components=2).fit_transform(X_multi)
    Y_tsne_gmina = TSNE(
        n_components=2,
        perplexity=35,
        init="pca",
        learning_rate="auto",
        max_iter=600,
        random_state=RANDOM_STATE,
    ).fit_transform(X_multi)

    embeddings_gmina = {"PCA": Y_pca_gmina, "t-SNE": Y_tsne_gmina}
    if UMAP_AVAILABLE:
        embeddings_gmina["UMAP"] = umap.UMAP(
            n_neighbors=25,
            min_dist=0.15,
            random_state=RANDOM_STATE,
        ).fit_transform(X_multi)

    color_value = spatial["target_spatial_hub_top25_degree50"]
    for method_name, coordinates in embeddings_gmina.items():
        plt.figure(figsize=(7, 5))
        plt.scatter(coordinates[:, 0], coordinates[:, 1], c=color_value, s=12, alpha=0.75)
        plt.title(f"{method_name}: typologia przestrzenna gmin; kolor = hub top 25%")
        plt.xlabel("wymiar 1")
        plt.ylabel("wymiar 2")
        plt.show()

    # Klasteryzacja i drzewo jako interpretacja klastrów.
    cluster_labels = KMeans(n_clusters=6, n_init=20, random_state=RANDOM_STATE).fit_predict(X_multi)
    surrogate_tree = DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=30,
        random_state=RANDOM_STATE,
    ).fit(feature_frame, cluster_labels)

    print(export_text(surrogate_tree, feature_names=feature_candidates))
else:
    print(
        "Rozszerzenie wielowymiarowe jest wyłączone. "
        "Po części obowiązkowej ustaw RUN_MULTIDIMENSIONAL_EXTENSION=True."
    )


# 16. Od screeningu do rzeczywistego planowania

Obecna analiza może wskazać miejsca wymagające dalszej weryfikacji, ale nie wystarcza do decyzji inwestycyjnej.

## Najważniejsze brakujące warstwy

| Warstwa | Po co? |
|---|---|
| populacja miejscowości/gmin | punkty PRNG nie są ważone liczbą mieszkańców |
| struktura wieku, szkoły, miejsca pracy | przybliżenie celów i częstotliwości podróży |
| istniejące przystanki i GTFS | ocena obecnej podaży transportu |
| sieć drogowa i czasy przejazdu | odległość haversine nie uwzględnia dróg |
| częstotliwość kursów | sam fakt istnienia połączenia nie oznacza dobrej obsługi |
| dane o potokach pasażerskich | rzeczywisty target popytu |
| samochody na gospodarstwo domowe | zależność od transportu publicznego |
| szpitale, szkoły średnie, kolej | konkretne cele dostępności |

## Naturalne rozszerzenie ML

Po dołączeniu zewnętrznego targetu można porównać:

- model bazowy: tylko demografia i gospodarka,
- model przestrzenny: cechy BDL + odległości,
- model grafowy: cechy BDL + centralności i spatial lag,
- Decision Tree / Random Forest / XGBoost.

Najciekawsze pytanie:

> Czy cechy grafowe poprawiają prognozę rzeczywistego popytu lub dostępności ponad zwykłe cechy tabelaryczne?## Wniosek metodologiczny z części drzewiastej

Płytkie drzewo może pełnić dwie role:

1. **surrogate** — streszcza reguły istniejącego wskaźnika,
2. **model kontekstowy** — sprawdza, czy szersze cechy przestrzenne pozwalają przewidzieć wynik bez znajomości jego bezpośrednich składników.

Dopiero model uczony na niezależnym, obserwowanym targecie może być traktowany jako model predykcyjny.

# 17. Szablon wniosków do uzupełnienia

Uzupełnij własnymi liczbami i nazwami.

## A. Graf miast

1. Dla $\varepsilon=\ldots$ graf ma $\ldots$ krawędzi i $\ldots$ składowych.
2. Największa składowa przekracza 90% miast przy około $\ldots$ km.
3. Najwyższy stopień binarny mają: $\ldots$.
4. Najwyższy stopień ważony mają: $\ldots$.
5. Różnica pomiędzy rankingami oznacza: $\ldots$.
6. DBSCAN przy `eps=...` znajduje $\ldots$ klastrów i $\ldots$% szumu.

## B. Screening połączeń

1. Wybrane województwo: $\ldots$.
2. Trzy gminy o najwyższym `need_score`: $\ldots$.
3. Najwyżej ocenione relacje gmina–miasto: $\ldots$.
4. Pierwsze 10 relacji obejmuje $\ldots$% miejscowości i $\ldots$% `burden`.
5. Wynik jest sensowny / niesensowny, ponieważ: $\ldots$.

## C. Ograniczenia

1. `need_score` mierzy: $\ldots$.
2. `need_score` nie mierzy: $\ldots$.
3. Najważniejsza brakująca zmienna: $\ldots$.
4. Następny krok analizy: $\ldots$.

## D. Jedna rekomendacja analityczna

> Na podstawie screeningu relacja/gmina $\ldots$ powinna zostać sprawdzona w pierwszej kolejności, ponieważ $\ldots$. Przed decyzją należy zweryfikować $\ldots$.


# 18. Automatyczny szkic wniosków z projektu

Po wykonaniu wszystkich `TODO` poniższa komórka tworzy szkic, który należy **sprawdzić i rozwinąć własną interpretacją**.


In [ ]:

top_region_gminas = (
    region_need.sort_values("need_score", ascending=False)
    .head(3)[["gmina", "need_score", "mean_km", "share_gt15"]]
)
top_region_routes = (
    region_relations.head(3)[[
        "gmina", "nearest_city_name", "route_priority", "mean_km", "n_settlements"
    ]]
)

print("=== GRAF MIAST ===")
print(f"Próg 50% największej składowej: {eps50:.1f} km")
print(f"Próg 90% największej składowej: {eps90:.1f} km")
print(f"Próg pełnej spójności: {eps_connected:.1f} km")
print("Najważniejszy obszar hubowy:", top_weighted_names)
print(
    "Porównanie grafów: epsilon ogranicza długość każdej krawędzi, "
    "natomiast kNN zapewnia sąsiadów, lecz w peryferiach może tworzyć długie połączenia."
)

print("\n=== SCREENING REGIONALNY ===")
print("Województwo:", SELECT_PROVINCE)
display(top_region_gminas.round(2))
display(top_region_routes.round(2))

if len(scenario) >= 10:
    row10 = scenario.loc[scenario["rank"].eq(10)].iloc[0]
    print(
        f"Pierwszych 10 relacji obejmuje {100 * row10['coverage_share']:.1f}% "
        f"miejscowości i {100 * row10['cumulative_burden_share']:.1f}% burden w regionie."
    )

print("\n=== CO WOLNO WNIOSKOWAĆ? ===")
print(
    "Wyniki wskazują progi, huby, peryferia i relacje do dalszej weryfikacji. "
    "Nie są jeszcze planem transportowym, ponieważ brakuje populacji, czasu przejazdu, "
    "istniejącej podaży GTFS i obserwowanych potoków pasażerskich."
)


# Checklista projektu

- [ ] policzono $D_{\text{city}}$,
- [ ] zbudowano $A_{\text{city}}$ i $W_{\text{city}}$,
- [ ] wykonano skan $\varepsilon$,
- [ ] porównano huby binarne i ważone,
- [ ] wykonano DBSCAN,
- [ ] sprawdzono opcjonalny Isomap miast lub opisano, dlaczego go pominięto,
- [ ] przypisano miejscowości do najbliższego miasta,
- [ ] policzono `need_score`,
- [ ] wybrano województwo,
- [ ] utworzono ranking relacji,
- [ ] zapisano minimum pięć wniosków,
- [ ] wskazano ograniczenia i brakujące dane.

Najważniejsza puenta:

> Graf nie podejmuje decyzji planistycznej. Pozwala jednak uporządkować relacje, wykryć progi, huby, peryferia i scenariusze, które następnie można zweryfikować lepszymi danymi.